# What iBART can (and can't) find

iBART discovers **nonlinear equations** -- that is its whole purpose. It builds
nonlinear descriptors by composing operators (`square, sqrt, log, exp, sin, cos,
abs, *, /, |a-b|`) on the inputs. The model is *linear in the coefficients on a
nonlinear basis* (like SINDy or polynomial regression): the discovered equation
is nonlinear, the fit is convex.

This notebook shows the wins, the one real limitation, and two workarounds.

## 1. Nonlinear expressions it recovers exactly

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from jax_ibart import IBart

rng = np.random.default_rng(0)
n = 200
x1, x2, x3 = (rng.uniform(0.1, 1, n) for _ in range(3))
X = np.column_stack([x1, x2, x3])
names = ["x1", "x2", "x3"]


def fit_and_score(y, schedule):
    m = IBart(backend="diff", schedule=schedule, max_dim=1, seed=0, steps=1500).fit(
        X, y, names
    )
    pred = m.predict(X)
    r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return m, pred, r2


cases = [
    ("sin(pi*x1*x2)", np.sin(np.pi * x1 * x2), ("b", "u")),
    ("(exp(x1)-exp(x2))^2", (np.exp(x1) - np.exp(x2)) ** 2, ("u", "b", "u")),
]
results = []
for label, y, sched in cases:
    m, pred, r2 = fit_and_score(y, sched)
    results.append((label, y, pred, m, r2))
    print(f"{label:>22}  ->  {m.descriptors_}   R2={r2:.4f}")

Parity plots: the recovered nonlinear models land on the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (label, y, pred, m, r2) in zip(axes, results):
    ax.scatter(y, pred, s=14, alpha=0.6)
    lo, hi = float(y.min()), float(y.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_title(f"y = {label}\nR$^2$={r2:.4f}")
    ax.set_xlabel("true y")
    ax.set_ylabel("predicted")
    ax.set_aspect("equal")
fig.tight_layout()
plt.show()

Both are recovered with R^2 = 1.0. Transcendental, rational, product, and
composed nonlinearities are all in scope as long as the operators can build them.

## 2. The one real limitation: free parameters *inside* a nonlinearity

iBART fits **linear coefficients** on a fixed, operator-generated library. It
cannot fit a *continuous parameter buried inside* a term unless that exact term
is generated. A power law with a non-integer exponent is the classic case: there
is no arbitrary-power operator, so `x^2.3` is only *approximated* (here by `x^2`).

In [ ]:
xp = rng.uniform(0.5, 3.0, n)
Xp = np.column_stack([xp, rng.uniform(0, 1, n)])
y = xp**2.3  # exponent 2.3 is not constructible
m = IBart(backend="diff", schedule=("u",), max_dim=1, seed=0, steps=1200).fit(
    Xp, y, ["x", "d"]
)
pr = m.predict(Xp)
print(
    "x^2.3  -> found",
    m.descriptors_,
    f" R2={1 - ((y-pr)**2).sum()/((y-y.mean())**2).sum():.4f}  (approximate, not exact)",
)

Plotting the fit against `x` shows the approximation drifting from the true curve:

In [ ]:
order = np.argsort(xp)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].plot(xp[order], y[order], lw=2, label="true  x$^{2.3}$")
axes[0].plot(xp[order], pr[order], "--", lw=2, label=f"iBART  {m.descriptors_[0]}")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()
axes[0].set_title("best the operator library can do")
axes[1].plot(xp[order], (pr - y)[order], color="C3")
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_xlabel("x")
axes[1].set_ylabel("residual (fit - true)")
axes[1].set_title("structured residual = missing exponent")
fig.tight_layout()
plt.show()

Same story for **Arrhenius** `k = A exp(-E/RT)` with a fitted activation energy
`E` *inside* the exponential, or **Michaelis-Menten** `v = Vmax S/(Km+S)` with a
fitted `Km` *in the denominator*. These are nonlinear *in the parameters*. Two
standard fixes:

### Workaround A -- linearize (Arrhenius)

In [ ]:
R = 8.314
A_true = 1e3
E_true = 50000.0
T = rng.uniform(300, 600, n)
k = A_true * np.exp(-E_true / (R * T))

# ln(k) = ln(A) - (E/R) * (1/T)  is LINEAR in 1/T
invT = 1.0 / T
slope, intercept = np.polyfit(invT, np.log(k), 1)
print(f"recovered E = {-slope * R:,.0f} J/mol   (true 50000)")
print(f"recovered A = {np.exp(intercept):.1f}      (true 1000)")

In [ ]:
o = np.argsort(invT)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].plot(T[np.argsort(T)], k[np.argsort(T)], lw=2)
axes[0].set_xlabel("T (K)")
axes[0].set_ylabel("k")
axes[0].set_title("k vs T: nonlinear in E (curved)")
axes[1].scatter(invT, np.log(k), s=12, alpha=0.6, label="data")
axes[1].plot(invT[o], (slope * invT + intercept)[o], "C3", lw=2, label="linear fit")
axes[1].set_xlabel("1 / T")
axes[1].set_ylabel("ln k")
axes[1].set_title(f"Arrhenius plot: slope = -E/R\nE = {-slope*R:,.0f} J/mol")
axes[1].legend()
fig.tight_layout()
plt.show()

Transforming to `ln k` vs `1/T` turns the curved, nonlinear-in-parameter problem
into a straight line whose slope gives `E`. iBART recovers the same structure
when given `1/T` (its `inv` operator) and the log-transformed target.

### Workaround B -- implicit / null-space form (Michaelis-Menten)

**What "null space" means, basically.** Usual regression fits `y = f(x)` -- one
column is the target, and we predict it from the others. But a constraint like
`x^2 + y^2 = 1` has *no* target: no single variable is "the output." Instead, a
particular weighted sum of the candidate terms is **always zero**.

So build a table whose columns are candidate terms (here `S, v, S^2, v^2, S*v`,
and a column of `1`s). We want weights `w` such that

    w1*S + w2*v + ... + w6*1  =  0   for every row.

That weight vector is called a **null-space vector** of the table: multiplying
the table by it gives all zeros. Standard regression can't find it (there's no
`y`); a singular value decomposition (SVD) can -- the direction the data *never*
varies along is exactly the constraint. The smallest singular value measures how
close to "exactly zero" it is (0 = an exact law).

For Michaelis-Menten, rearranging `v = Vmax*S/(Km+S)` gives
`Vmax*S - Km*v - S*v = 0` -- a sum of terms that is always zero, i.e. a
null-space vector of the `[S, v, S*v, ...]` table. That is what we recover:

In [ ]:
from jax_ibart.dynamics import implicit_constraints
import sympy as sp

Vmax, Km = 5.0, 2.0
S = rng.uniform(0.1, 10.0, n)
v = Vmax * S / (Km + S)

# v(Km + S) = Vmax S  ->  Vmax*S - Km*v - S*v = 0  : an algebraic constraint
res = implicit_constraints(np.column_stack([S, v]), ["S", "v"], degree=2, tol=1e-2)
print("recovered constraint:", sp.nsimplify(res["constraint"], rational=False).evalf(3))
print("residual (0 = exact):", round(res["residual"], 6))
print("\ntrue: 5*S - 2*v - S*v = 0  ->  divide by 5:  S - 0.4*v - 0.2*S*v = 0")

Reconstructing `v(S)` from the recovered coefficients reproduces the data:

In [ ]:
# constraint:  cS*S + cv*v + cSv*S*v + ... = 0  ->  solve for v(S)
# coef vector aligns with res['terms']; pull the S, v, S*v weights
terms = res["terms"]
w = res["coef"]


def wof(name):
    return w[terms.index(name)]


cS, cv, cSv = wof("S"), wof("v"), wof("S*v")
# cS*S + cv*v + cSv*S*v = 0  ->  v = -cS*S / (cv + cSv*S)
Sg = np.linspace(0.1, 10, 100)
v_fit = -cS * Sg / (cv + cSv * Sg)
Vmax_hat, Km_hat = -cS / cSv, cv / cSv

fig, ax = plt.subplots(figsize=(5, 3.6))
ax.scatter(S, v, s=14, alpha=0.5, label="data")
ax.plot(
    Sg, v_fit, "C3", lw=2, label=f"recovered (Vmax={Vmax_hat:.2f}, Km={Km_hat:.2f})"
)
ax.set_xlabel("S")
ax.set_ylabel("v")
ax.set_title("Michaelis-Menten via null-space form")
ax.legend()
fig.tight_layout()
plt.show()

The implicit (null-space) formulation recovers the rational Michaelis-Menten
kinetics **exactly** as an algebraic constraint, from which `Vmax` and `Km` read
straight off the coefficients (here Vmax=5, Km=2). This is the same machinery
that underlies the DAE foundation (`implicit_constraints`) and SINDy-PI (see
[references](../references.md)).

**Summary:** nonlinear *expressions* -- yes, that is the point. Free parameters
*inside* a nonlinearity -- only via the operator library, a linearizing
transform, or the implicit null-space form.